# Field classification: strata and pattern (donor) fields

Assign fields to groundwater-access strata (REM + stream type) and identify non-irrigated pattern fields for ET_gwsm donor matching.

In [ ]:
# Shared color palette — used throughout all notebooks
STREAM_BLUE = "#2171b5"
WET_CMAP = "Blues"
TERRAIN_CMAP = "terrain"
REM_CMAP = "RdYlBu_r"
NDWI_CMAP = "RdYlGn"
STRATA_COLORS = {
    "perennial": "#2166ac",
    "intermittent": "#74add1",
    "managed": "#4dac26",
    "non_partitioned": "#d9d9d9",
}
PARTITION_COLORS = {"pe": "#a6cee3", "et_gwsm": "#1f78b4", "et_irr": "#e31a1c"}

## 1. Why stratify fields?

Partitioning depends on baseline non-irrigation ET in shallow-GW settings. Handily uses two controls:

1. `rem_mean` threshold (partitioned vs non-partitioned)
2. Nearest stream type from NHD (perennial/intermittent/managed)

Strata used in this example:

| Strata | Definition | Notes |
|--------|------------|------|
| `perennial` | low REM + nearest stream perennial | high baseline ET potential |
| `intermittent` | low REM + intermittent/ephemeral | seasonal baseline |
| `managed` | low REM + canal/ditch | anthropogenic water proximity |
| `non_partitioned` | high REM | minimal shallow-GW influence |

## 2. Setup

In [1]:
import os
import tomllib
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from handily.config import HandilyConfig
from handily.nhd import classify_flowlines, FCODE_CATEGORIES

# Load configuration
config_path = Path("beaverhead_config.toml")
with open(config_path, "rb") as f:
    cfg = tomllib.load(f)

config = HandilyConfig.from_dict(cfg)
out_dir = os.path.expanduser(config.out_dir)

print(f"Output directory: {out_dir}")

Output directory: /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/


## 3. REM-based partitioning

Default: `rem_mean < 2.0` ⇒ partitioned. Treat this as a tunable heuristic (soil texture/capillary fringe, ditch depth, DEM/NDWI quality).

In [2]:
# Load fields with REM statistics
fields_path = os.path.join(out_dir, "fields_bounds.fgb")

if os.path.exists(fields_path):
    fields = gpd.read_file(fields_path)
    
    if 'rem_mean' in fields.columns:
        # Apply threshold
        threshold = cfg.get("rem_threshold", 2.0)
        fields['partitioned'] = fields['rem_mean'] < threshold
        
        n_partitioned = fields['partitioned'].sum()
        n_non_part = (~fields['partitioned']).sum()
        
        print(f"REM threshold: {threshold}m")
        print(f"Partitioned (REM < {threshold}m): {n_partitioned} fields")
        print(f"Non-partitioned (REM >= {threshold}m): {n_non_part} fields")
        
        # Visualize
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Plot non-partitioned first (gray)
        fields[~fields['partitioned']].plot(
            ax=ax, color='lightgray', edgecolor='black', linewidth=0.3,
            label=f'Non-partitioned (n={n_non_part})'
        )
        # Plot partitioned on top (blue)
        fields[fields['partitioned']].plot(
            ax=ax, color='steelblue', edgecolor='black', linewidth=0.3,
            label=f'Partitioned (n={n_partitioned})'
        )
        
        ax.set_title(f"REM-Based Partitioning (threshold = {threshold}m)")
        ax.legend()
        plt.tight_layout()
else:
    print(f"Fields not found at {fields_path}")
    print("Run the REM workflow first.")

Fields not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/fields_bounds.fgb
Run the REM workflow first.


In [ ]:
# Threshold sensitivity analysis (skipped if fields data not yet available)
fields = None  # initialize to None in case REM workflow hasn't run yet
fields_path_check = os.path.join(out_dir, "fields_bounds.fgb")
if os.path.exists(fields_path_check):
    import geopandas as gpd
    fields = gpd.read_file(fields_path_check)

if fields is not None and 'rem_mean' in fields.columns:
    thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
    partitioned_counts = []
    for t in thresholds:
        n = (fields['rem_mean'] < t).sum()
        partitioned_counts.append(n)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(thresholds, partitioned_counts, width=0.4, edgecolor='black')
    ax.axvline(2.0, color='red', linestyle='--', label='Default (2m)')
    ax.set_xlabel('REM Threshold (m)')
    ax.set_ylabel('Number of Partitioned Fields')
    ax.set_title('Threshold Sensitivity Analysis')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] fields data not available — run the REM workflow first")

In [3]:
if 'rem_mean' in fields.columns:
    thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
    partitioned_counts = []
    
    for t in thresholds:
        n = (fields['rem_mean'] < t).sum()
        partitioned_counts.append(n)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(thresholds, partitioned_counts, width=0.4, edgecolor='black')
    ax.axvline(2.0, color='red', linestyle='--', label='Default (2m)')
    ax.set_xlabel('REM Threshold (m)')
    ax.set_ylabel('Number of Partitioned Fields')
    ax.set_title('Threshold Sensitivity Analysis')
    ax.legend()
    plt.tight_layout()

NameError: name 'fields' is not defined

## 4. Stream classification

Handily maps NHD FCODE values to three stream-type classes:

| Category | FCODE Values | Description |
|----------|--------------|-------------|
| `perennial` | 46006 | Natural perennial streams/rivers |
| `intermittent` | 46003, 46007, 46000 | Intermittent, ephemeral, or unknown |
| `managed` | 33400, 33600-33603, 46800 | Canals, ditches, artificial paths |

In [ ]:
# FCODE category bar chart
import collections
from handily.nhd import FCODE_CATEGORIES

flowlines_path = os.path.join(out_dir, "flowlines_bounds.fgb")
if os.path.exists(flowlines_path):
    flowlines_gdf = gpd.read_file(flowlines_path)
    fcode_col = "fcode" if "fcode" in flowlines_gdf.columns else (
        "FCODE" if "FCODE" in flowlines_gdf.columns else None
    )
    if fcode_col:
        cats = [FCODE_CATEGORIES.get(fc, "unknown") for fc in flowlines_gdf[fcode_col]]
        counts = collections.Counter(c for c in cats if c is not None)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(
            counts.keys(), counts.values(),
            color=[STRATA_COLORS.get(k, "#aaa") for k in counts],
        )
        ax.set_title("NHD Flowline FCODE Categories")
        ax.set_ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("[SKIP] No fcode/FCODE column found in flowlines")
else:
    print("[SKIP] Flowlines not yet computed — run REMWorkflow first")

In [ ]:
# Nearest stream type map + stream distance histogram
from handily.nhd import assign_nearest_stream_type

stratified_path = os.path.join(out_dir, "fields_stratified.fgb")
flowlines_path = os.path.join(out_dir, "flowlines_bounds.fgb")

if os.path.exists(stratified_path) and os.path.exists(flowlines_path):
    fields_strat = gpd.read_file(stratified_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: fields colored by nearest_stream_type
    if "nearest_stream_type" in fields_strat.columns:
        for stype, color in STRATA_COLORS.items():
            mask = fields_strat["nearest_stream_type"] == stype
            if mask.any():
                fields_strat[mask].plot(ax=axes[0], color=color,
                                        edgecolor="black", linewidth=0.3,
                                        label=f"{stype} (n={mask.sum()})")
        axes[0].set_title("Nearest Stream Type")
        axes[0].legend(fontsize=8)
    else:
        fields_strat.plot(ax=axes[0], color="#ccc", edgecolor="black", linewidth=0.3)
        axes[0].set_title("Fields (nearest_stream_type not yet assigned)")
    
    # Right: stream distance histogram
    if "stream_distance" in fields_strat.columns:
        axes[1].hist(fields_strat["stream_distance"].dropna(), bins=30,
                     color=STREAM_BLUE, edgecolor="none", alpha=0.7)
        med = float(fields_strat["stream_distance"].median())
        axes[1].axvline(med, color="red", linestyle="--", label=f"Median: {med:.0f} m")
        axes[1].set_xlabel("Distance to nearest stream (m)")
        axes[1].set_ylabel("Field count")
        axes[1].set_title("Stream Distance Distribution")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "stream_distance column\nnot found",
                     ha="center", va="center", transform=axes[1].transAxes)
        axes[1].set_title("Stream Distance")
    
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Stratified fields not yet computed — run stratification first")

In [ ]:
# Load stratified fields + strata map with count bar chart
stratified_path = os.path.join(out_dir, "fields_stratified.fgb")

if os.path.exists(stratified_path):
    fields_stratified = gpd.read_file(stratified_path)
    
    if 'strata' in fields_stratified.columns:
        print("Strata distribution:")
        print(fields_stratified['strata'].value_counts())
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        
        # Left: choropleth map
        for strata, color in STRATA_COLORS.items():
            subset = fields_stratified[fields_stratified['strata'] == strata]
            if len(subset) > 0:
                subset.plot(
                    ax=axes[0], color=color, edgecolor='black', linewidth=0.3,
                    label=f'{strata} (n={len(subset)})'
                )
        axes[0].set_title("Fields by Strata")
        axes[0].legend(fontsize=8)
        
        # Right: count bar chart
        counts = fields_stratified['strata'].value_counts()
        axes[1].bar(
            counts.index, counts.values,
            color=[STRATA_COLORS.get(s, "#aaa") for s in counts.index],
            edgecolor="black",
        )
        axes[1].set_title("Field Count by Strata")
        axes[1].set_ylabel("Number of fields")
        axes[1].set_xlabel("Strata")
        
        plt.tight_layout()
        plt.show()
    else:
        print("'strata' column not found. Run stratification.")
else:
    print(f"Stratified fields not found at {stratified_path}")
    print("Run the stratification step.")

In [ ]:
# Load and classify flowlines
flowlines_path = os.path.join(out_dir, "flowlines_bounds.fgb")

if os.path.exists(flowlines_path):
    flowlines = gpd.read_file(flowlines_path)
    flowlines_classified = classify_flowlines(flowlines)
    
    print("Flowline categories:")
    print(flowlines_classified['stream_category'].value_counts())
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 10))
    
    colors = {'perennial': 'blue', 'intermittent': 'cyan', 'managed': 'orange'}
    
    for category, color in colors.items():
        subset = flowlines_classified[flowlines_classified['stream_category'] == category]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, linewidth=1.0, label=f'{category} (n={len(subset)})')
    
    ax.set_title("Flowlines by Category")
    ax.legend()
    plt.tight_layout()
else:
    print(f"Flowlines not found at {flowlines_path}")

> **Operational Step — Bucket Sync (IrrMapper)**
>
> IrrMapper results are exported to GCS and synced locally before pattern selection:
>
> ```python
> from handily.bucket import sync_bucket_to_local
> result = sync_bucket_to_local(
>     bucket="wudr",
>     bucket_prefix="handily",
>     local_root=config.local_data_root,
>     subdir="irrmapper",
>     glob_pattern="irr_freq",
>     dry_run=True,
> )
> print(f"Would copy {len(result.get('files', []))} files")
> ```
>
> CLI equivalent: `handily sync --config beaverhead_config.toml --subdir irrmapper`

## 6. IrrMapper: irrigation history

IrrMapper is an annual Landsat irrigation classification for the western US (1987–present). For each field we summarize irrigation frequency/probability (`irr_count`, `irr_freq`, `irr_mean`) for pattern selection.

In [ ]:
# Show bucket/local path configuration
print("=== Bucket/Local Mirror Configuration ===")
print(f"Bucket: {config.et_bucket}")
print(f"Bucket prefix: {config.bucket_prefix}")
print(f"Project name: {config.project_name}")
print(f"Local data root: {config.local_data_root}")

# Show the mirrored paths
if config.et_bucket and config.local_data_root:
    bucket_path = f"gs://{config.et_bucket}/{config.bucket_prefix}/{config.project_name}/irrmapper/"
    local_path = config.get_local_path("irrmapper")
    
    print("\nIrrMapper paths:")
    print(f"  Bucket: {bucket_path}")
    print(f"  Local:  {local_path}")

### Sync

Sync utilities live in `handily.bucket` and are exposed via `handily sync`.

In [ ]:
# Dry run to see what would be synced (uncomment to run)
# from handily.bucket import sync_bucket_to_local
#
# if config.et_bucket and config.local_data_root:
#     full_prefix = f"{config.bucket_prefix}/{config.project_name}"
#     
#     result = sync_bucket_to_local(
#         bucket=config.et_bucket,
#         bucket_prefix=full_prefix,
#         local_root=config.local_data_root,
#         subdir="irrmapper",
#         glob_pattern="*irr_freq*",
#         dry_run=True,  # Preview only
#     )
#     print(f"Would copy {len(result['files'])} files")

print("Uncomment the code above to run a dry-run sync.")

## 8. Loading IrrMapper Data

## 9. Pattern field selection

Pattern fields are consistently non-irrigated donors used to estimate `ET_gwsm` within strata. In this example:

- `irr_freq <= 0.1`
- `irr_mean <= 0.05`

For each irrigated field, the nearest in-stratum pattern field is selected as the donor.

In [ ]:
# Load fields with pattern column if available
pattern_path = os.path.join(out_dir, "fields_pattern.fgb")

if os.path.exists(pattern_path):
    fields_pattern = gpd.read_file(pattern_path)
    
    if 'pattern' in fields_pattern.columns:
        n_pattern = fields_pattern['pattern'].sum()
        n_irrigated = (~fields_pattern['pattern']).sum()
        
        print(f"Pattern fields: {n_pattern}")
        print(f"Irrigated fields: {n_irrigated}")
        
        if 'strata' in fields_pattern.columns:
            print("\nPattern fields by strata:")
            print(fields_pattern.groupby('strata')['pattern'].sum())
        
        # Visualize
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Plot irrigated fields (gray)
        fields_pattern[~fields_pattern['pattern']].plot(
            ax=ax, color='lightgray', edgecolor='black', linewidth=0.3,
            label=f'Irrigated (n={n_irrigated})'
        )
        # Plot pattern fields (green)
        fields_pattern[fields_pattern['pattern']].plot(
            ax=ax, color='forestgreen', edgecolor='black', linewidth=0.3,
            label=f'Pattern (n={n_pattern})'
        )
        
        ax.set_title("Pattern Field Selection")
        ax.legend()
        plt.tight_layout()
else:
    print(f"Pattern fields not found at {pattern_path}")
    print("Run the pattern selection step.")

## 10. Running stratification

Run the stratification + pattern selection workflow.

In [ ]:
# irr_freq vs irr_mean scatter with classification cutoff lines
irrmapper_csv = cfg.get('irrmapper_csv')
irrmapper_df = None
if irrmapper_csv and os.path.exists(os.path.expanduser(irrmapper_csv)):
    irrmapper_df = pd.read_csv(os.path.expanduser(irrmapper_csv))

if irrmapper_df is not None and "irr_freq" in irrmapper_df.columns and "irr_mean" in irrmapper_df.columns:
    pattern_mask = irrmapper_df.get("pattern", pd.Series([False] * len(irrmapper_df)))
    colors = pattern_mask.map({True: "green", False: "gray", 1: "green", 0: "gray"})
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(irrmapper_df["irr_freq"], irrmapper_df["irr_mean"],
               c=colors, alpha=0.6, s=15)
    ax.axvline(0.1, color="red", linestyle="--", label="max_irr_freq=0.10")
    ax.axhline(0.05, color="orange", linestyle="--", label="max_irr_mean=0.05")
    ax.set_xlabel("Irrigation Frequency")
    ax.set_ylabel("Mean Irrigation")
    ax.set_title("IrrMapper Pattern Candidates (green = pattern donor)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] IrrMapper data not available — run EE export and sync first")

In [ ]:
# Example: Running stratification
# (Commented out to avoid re-running if already complete)

# from handily.stratify import stratify
#
# # Load required data
# rem_da = rxr.open_rasterio(os.path.join(out_dir, "rem_bounds.tif")).squeeze("band", drop=True)
# flowlines = gpd.read_file(os.path.join(out_dir, "flowlines_bounds.fgb"))
# fields = gpd.read_file(os.path.join(out_dir, "fields_bounds.fgb"))
#
# # Run stratification
# fields_stratified = stratify(
#     fields=fields,
#     flowlines=flowlines,
#     rem_da=rem_da,
#     rem_threshold=2.0,
#     max_stream_distance=None  # No distance cutoff
# )
#
# # Save result
# fields_stratified.to_file(os.path.join(out_dir, "fields_stratified.fgb"), driver="FlatGeobuf")
#
# print("Stratification complete!")
# print(fields_stratified['strata'].value_counts())

print("See beaverhead.py for the full workflow script.")

In [ ]:
# Donor readiness summary: pattern donors per strata
pattern_path = os.path.join(out_dir, "fields_pattern.fgb")
if os.path.exists(pattern_path):
    fields_pattern = gpd.read_file(pattern_path)
    if "strata" in fields_pattern.columns and "pattern" in fields_pattern.columns:
        from IPython.display import display
        summary = fields_pattern.groupby("strata")["pattern"].sum().reset_index()
        summary.columns = ["Strata", "Pattern Donors"]
        total = fields_pattern["strata"].value_counts().reset_index()
        total.columns = ["Strata", "Total Fields"]
        summary = summary.merge(total, on="Strata")
        summary["Donor %"] = (100 * summary["Pattern Donors"] / summary["Total Fields"]).round(1)
        display(summary)
    else:
        print("Required columns not found in pattern fields")
else:
    print("[SKIP] Pattern fields not yet computed — run pattern selection first")

### Output Files

| File | Location | Description |
|------|----------|-------------|
| `fields_stratified.fgb` | `out_dir` | Fields with `partitioned`, `nearest_stream_type`, `strata` columns |
| `fields_pattern.fgb` | `out_dir` | Fields with additional `pattern` column |
| `*_irr_freq.csv` | Bucket/Local mirror | IrrMapper export (from Earth Engine) |

## Key takeaways

- Strata constrain donor matching to comparable groundwater-access setting (REM) and water-source type.
- IrrMapper provides an empirical screen for stable non-irrigated pattern fields.
- Bucket mirroring + `handily sync` keeps Earth Engine exports consistent with local paths.

**Next**: [Notebook 04 - Climate and ET](04_climate_and_et.ipynb)

> **Repo Capability — Notebook 3 (Field Classification)**
> Modules exercised: `handily.stratify`, `handily.nhd`, `handily.pattern`, `handily.bucket`, `handily.et.irrmapper`
> Key functions: `stratify()`, `assign_strata()`, `classify_flowlines()`, `assign_nearest_stream_type()`, `assign_pattern_from_irrmapper()`, `sync_bucket_to_local()`
> CLI equivalents: `handily sync --subdir irrmapper`

> **Artifacts Produced**
> - `{out_dir}/fields_stratified.fgb` — field polygons with `strata`, `nearest_stream_type`, `stream_distance` columns
> - `{out_dir}/fields_pattern.fgb` — field polygons with additional `pattern` boolean column